In [ ]:
# 2월 15일 11시 5분 시작

# 원천 데이터 파일 경로는 기존 코드에서 쓰던 구조를 최대한 따르되, 혹시 네 로컬에 디렉토리명이 다르면 RAW_INPUTS만 한 번 수정하면 전체가 돌아가게 만들었다

In [1]:
# =========================================
# Notebook 01 — Data Ingestion + Sampling
# (revised to auto-detect 3 input CSVs and enforce patent_id as string)
# =========================================
# INPUT (must exist in current working directory):
#   independent_claims_y2005_with_abstract_wipo_cpc.csv
#   independent_claims_y2015_with_abstract_wipo_cpc.csv
#   independent_claims_y2025_with_abstract_wipo_cpc.csv
#
# OUTPUT (auto-created):
#   2nd_exp/
#     data/{year}/sampled_patents.csv
#     config/notebook01_config.json
#     logs/notebook01_log.txt
#
# Notes:
# - patent_id is strictly normalized to string to avoid matching failures.
# - claims / abstract columns are auto-mapped if names differ.
# - sample size per year fixed to 1050 (as decided).

import os
import re
import json
import time
import random
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# ----------------------------
# 0) Fixed design parameters
# ----------------------------
SEED = 42
SAMPLE_PER_YEAR = 1050
YEARS = [2005, 2015, 2025]

INPUT_FILES = {
    2005: "independent_claims_y2005_with_abstract_wipo_cpc.csv",
    2015: "independent_claims_y2015_with_abstract_wipo_cpc.csv",
    2025: "independent_claims_y2025_with_abstract_wipo_cpc.csv",
}

BASE_DIR = "2nd_exp"
DATA_DIR = os.path.join(BASE_DIR, "data")
CFG_DIR  = os.path.join(BASE_DIR, "config")
LOG_DIR  = os.path.join(BASE_DIR, "logs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CFG_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

LOG_PATH = os.path.join(LOG_DIR, "notebook01_log.txt")

random.seed(SEED)
np.random.seed(SEED)

def log(msg: str):
    print(msg)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(msg + "\n")

# reset log
with open(LOG_PATH, "w", encoding="utf-8") as f:
    f.write("")

log("=== Notebook 01 started ===")
log(f"SEED={SEED}, SAMPLE_PER_YEAR={SAMPLE_PER_YEAR}")
log(f"Working dir: {os.getcwd()}")

# ----------------------------
# 1) Utilities: column mapping & patent_id enforcement
# ----------------------------
def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def find_first_existing_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

def enforce_patent_id_str(df: pd.DataFrame, year: int) -> pd.DataFrame:
    """
    Make sure df['patent_id'] exists and is a normalized string.
    This prevents mixed dtype (int/float/str) issues that break matching.
    """
    # If patent_id doesn't exist, try common id columns
    if "patent_id" not in df.columns:
        id_col = find_first_existing_col(df, [
            "patent_id",
            "publication_number",
            "pub_number",
            "publicationno",
            "publication_no",
            "publication",
            "doc_number",
            "docno",
            "appln_id",
            "application_id",
            "patent_number",
            "patent_no",
            "pn",
            "id",
        ])
        if id_col is not None:
            df = df.rename(columns={id_col: "patent_id"})
            log(f"[{year}] patent_id column mapped from '{id_col}'.")
        else:
            # Create a stable fallback id
            df["patent_id"] = [f"{year}_{i}" for i in range(len(df))]
            log(f"[{year}] patent_id column not found. Created fallback ids 'YEAR_index'.")

    # Force string — IMPORTANT
    # 1) Convert to string, 2) strip spaces, 3) remove ".0" if it came from float-like strings, 4) keep only as-is otherwise
    pid = df["patent_id"].astype(str)

    # Normalize whitespace
    pid = pid.str.strip()

    # Handle common corruption: "12345.0" from float parsing earlier
    pid = pid.str.replace(r"\.0$", "", regex=True)

    # Replace "nan", "None", "" with fallback ids
    bad = pid.isna() | (pid.str.lower().isin(["nan", "none"])) | (pid.str.len() == 0)
    if bad.any():
        idxs = np.where(bad.values)[0].tolist()
        for i in idxs[:5]:
            log(f"[{year}] WARNING: bad patent_id at row {i} -> replaced with fallback.")
        pid.loc[bad] = [f"{year}_missing_{i}" for i in range(bad.sum())]

    df["patent_id"] = pid

    # Ensure uniqueness: if duplicates exist, suffix them deterministically
    if df["patent_id"].duplicated().any():
        log(f"[{year}] WARNING: duplicate patent_id detected. Applying deterministic suffixing.")
        # deterministic suffix based on row order
        dup_counts = {}
        new_ids = []
        for x in df["patent_id"].tolist():
            if x not in dup_counts:
                dup_counts[x] = 0
                new_ids.append(x)
            else:
                dup_counts[x] += 1
                new_ids.append(f"{x}__dup{dup_counts[x]}")
        df["patent_id"] = pd.Series(new_ids, dtype=str)

    # final assert
    assert df["patent_id"].dtype == object, "patent_id must be object dtype (string-like)."
    return df

def map_claims_abstract_cols(df: pd.DataFrame, year: int) -> pd.DataFrame:
    """
    Map columns to canonical:
      - claims
      - abstract
    We keep other columns as-is.
    """
    # claims candidates
    claims_col = find_first_existing_col(df, [
        "claims", "independent_claims", "independent_claim", "claim", "claims_text",
        "ind_claims", "ind_claim", "independentclaims"
    ])
    if claims_col is None:
        raise ValueError(f"[{year}] Cannot find claims column. Available columns: {list(df.columns)[:50]} ...")

    # abstract candidates
    abstract_col = find_first_existing_col(df, ["abstract", "abstract_text", "abs"])
    if abstract_col is None:
        raise ValueError(f"[{year}] Cannot find abstract column. Available columns: {list(df.columns)[:50]} ...")

    if claims_col != "claims":
        df = df.rename(columns={claims_col: "claims"})
        log(f"[{year}] claims column mapped from '{claims_col}'.")
    if abstract_col != "abstract":
        df = df.rename(columns={abstract_col: "abstract"})
        log(f"[{year}] abstract column mapped from '{abstract_col}'.")

    # normalize text
    df["claims"] = df["claims"].fillna("").map(normalize_text)
    df["abstract"] = df["abstract"].fillna("").map(normalize_text)

    return df

# ----------------------------
# 2) Load, normalize, sample, save
# ----------------------------
def load_year_csv(year: int, filename: str) -> pd.DataFrame:
    path = os.path.join(os.getcwd(), filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"[{year}] Input file not found in working directory: {filename}")

    log(f"[{year}] Loading: {filename}")

    # 핵심: dtype=str로 강제해서 mixed dtype을 원천 차단
    # keep_default_na=False: 문자열 "NA" 등을 NaN으로 바꾸는 것을 줄여줌(원문 보존 성향)
    df = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        na_values=[],
        low_memory=False
    )
    log(f"[{year}] Loaded rows={len(df):,}, cols={len(df.columns)}")
    return df

def sample_df(df: pd.DataFrame, year: int, n: int) -> pd.DataFrame:
    if len(df) < n:
        log(f"[{year}] WARNING: rows ({len(df)}) < sample size ({n}). Using all rows.")
        return df.copy()
    # deterministic sample by SEED, but year-specific to avoid identical draws across years
    rng = np.random.default_rng(SEED + year)
    idx = rng.choice(len(df), size=n, replace=False)
    return df.iloc[idx].copy()

# record config
cfg = {
    "seed": SEED,
    "sample_per_year": SAMPLE_PER_YEAR,
    "years": YEARS,
    "input_files": INPUT_FILES,
    "outputs": {
        "sampled_patents_path": "2nd_exp/data/{year}/sampled_patents.csv"
    }
}

for year in YEARS:
    fname = INPUT_FILES[year]
    df = load_year_csv(year, fname)

    # canonical mapping & strict id handling
    df = map_claims_abstract_cols(df, year)
    df = enforce_patent_id_str(df, year)

    # optional: keep CPC/WIPO columns if exist (no need to rename; keep as-is)
    # but ensure consistent dtypes (string)
    for c in df.columns:
        if c not in ["claims", "abstract", "patent_id"]:
            # keep as str, normalize whitespace lightly
            df[c] = df[c].astype(str).str.strip()

    # sample 1050
    df_s = sample_df(df, year, SAMPLE_PER_YEAR)

    # final dtypes check
    df_s["patent_id"] = df_s["patent_id"].astype(str).str.strip()
    assert df_s["patent_id"].apply(lambda x: isinstance(x, str)).all(), f"[{year}] patent_id not all strings!"

    # save to standard path
    out_dir = os.path.join(DATA_DIR, str(year))
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "sampled_patents.csv")
    df_s.to_csv(out_path, index=False, encoding="utf-8")
    log(f"[{year}] Saved sampled_patents.csv -> {out_path} (rows={len(df_s):,})")

# save config
cfg_path = os.path.join(CFG_DIR, "notebook01_config.json")
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)
log(f"Saved config -> {cfg_path}")

log("=== Notebook 01 completed successfully ===")

print("\nDone. Next: run Notebook 02 (query generation + leakage filtering).")
print("Outputs created under:", BASE_DIR)


=== Notebook 01 started ===
SEED=42, SAMPLE_PER_YEAR=1050
Working dir: c:\pydir113_bgem3
[2005] Loading: independent_claims_y2005_with_abstract_wipo_cpc.csv
[2005] Loaded rows=141,170, cols=5
[2005] Saved sampled_patents.csv -> 2nd_exp\data\2005\sampled_patents.csv (rows=1,050)
[2015] Loading: independent_claims_y2015_with_abstract_wipo_cpc.csv
[2015] Loaded rows=292,573, cols=5
[2015] Saved sampled_patents.csv -> 2nd_exp\data\2015\sampled_patents.csv (rows=1,050)
[2025] Loading: independent_claims_y2025_with_abstract_wipo_cpc.csv
[2025] Loaded rows=240,989, cols=5
[2025] Saved sampled_patents.csv -> 2nd_exp\data\2025\sampled_patents.csv (rows=1,050)
Saved config -> 2nd_exp\config\notebook01_config.json
=== Notebook 01 completed successfully ===

Done. Next: run Notebook 02 (query generation + leakage filtering).
Outputs created under: 2nd_exp
